# Book 3 — Ranked Fusion Recommender

**"Readers also read…": a Goodreads book recommender** · SIADS 593 Milestone I, Fall 2026

Book 1 produced the 50,000-reader sample, the split by reader and the recommendable catalogue.
Book 2 built three recommenders on the training readers, each answering a work with its top five
`work_id`s, and saved their neighbour tables. This notebook combines the three answers into one,
checks how much the models agree, tunes the combination on validation readers, scores it once on
test readers next to the single models and the baselines, and defines the function the dashboard
calls: the two Jaccard models fused, after validation showed the directional score pulls the fused
list toward bestsellers.

## Method card

| | |
|---|---|
| **Inputs** | three ranked lists of five `work_id`s for the selected work: directional (likes), Jaccard (likes), finished together (shelves) |
| **Rule** | reciprocal rank fusion: `fused(w) = Σ 1 / (60 + rank of w in each list)`; works a model did not return contribute nothing |
| **Output** | `recommend_fused(work_id, n=5)`: the five works with the highest fused score |
| **Why ranks, not scores** | the three scores are on different scales: a share of readers (0–100%), a 0–1 overlap on likes, and a 0–1 overlap on finished books. Positions compare; the numbers do not. |
| **Parameters** | `k = 60`, equal weights, five candidates from each model; the recommender fuses the two Jaccard lists and leaves the directional list out (Section 3) |

**How the notebook runs.** The neighbour tables come from Book 2 (`data/processed/neighbours_*.pkl`).
Evaluation tables are saved as `data/processed/book3_*.pkl` and loaded on later runs; the first run
takes about ten minutes, later runs under one. `SEED = 593` throughout.

In [1]:
# Setup: Book 1 and Book 2 outputs, the project modules, and the catalogue every recommender draws from.
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

alt.renderers.enable("mimetype")

PROJECT = Path.cwd().resolve().parent if Path.cwd().name == "book3" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT / "scripts"))

import cooccurrence as co          # NeighbourTable and the recommendable catalogue
import evaluation as ev            # the evaluation protocol shared with Book 2
import fusion as fu                # the fused recommender, its variants, agreement and examples
import fusion_charts as fc         # the charts in Section 6

PROCESSED = PROJECT / "data" / "processed"
pd.set_option("display.width", 200)
SEED, MIN_LIKES, TOP_N, RRF_K = 593, 10, 5, 60

interactions = pd.read_pickle(PROCESSED / "interactions_sample_50k.pkl")
assignment = pd.read_pickle(PROCESSED / "user_splits.pkl")
books = pd.read_pickle(PROCESSED / "book_dimension.pkl")
train_likes = pd.read_pickle(PROCESSED / "train_likes_per_work.pkl")
similar_books = np.load(PROCESSED / "similar_books.npy")

interactions["split"] = interactions["user_id"].map(assignment)
train = interactions[interactions["split"] == "train"]
work_ids = co.recommendable_works(train_likes, floor=MIN_LIKES)
titles = books.set_index("work_id")["title"]
popularity = ev.popularity_from(train)

tables = fu.load_tables(PROCESSED)
print({name: len(table) for name, table in tables.items()})

{'Directional (likes)': 43203, 'Jaccard (likes)': 43203, 'Finished together (shelves)': 43203}


## 1 · The rule

Reciprocal rank fusion gives each work a score from its positions in the lists that returned it:

`fused(w) = Σ over models  1 / (k + rank of w in that model)`

with `k = 60`, the default proposed by Cormack, Clarke and Buettcher (2009). A work returned by two
models at rank 1 scores about twice one returned by a single model at rank 1, so the rule rewards
agreement. A smaller `k` would trust each model's first pick more; a larger `k` would flatten the
curve so that only agreement mattered. Ties break on the number of models that returned the work,
then the best single rank, then the lower `work_id`, so the output is deterministic.

The project has a verified implementation (`scripts/rank_fusion.py`, checked against a brute-force
reference). `scripts/fusion.py` uses a lighter version of the same arithmetic at query time, because
the evaluation asks tens of thousands of questions, and asserts that the two agree.

### Worked example

Three lists for one seed: A = [1, 2, 3], B = [2, 3, 4], C = [3, 1, 5]. By hand, with `k = 60`:

| work | positions | score |
|---|---|---|
| 3 | A3, B2, C1 | 1/63 + 1/62 + 1/61 = 0.0484 |
| 1 | A1, C2 | 1/61 + 1/62 = 0.0325 |
| 2 | A2, B1 | 1/62 + 1/61 = 0.0325 |
| 4 | B3 | 1/63 = 0.0159 |
| 5 | C3 | 1/63 = 0.0159 |

Work 3 wins because all three lists contain it. Works 1 and 2 tie on score and on the number of
models and on best rank, so the lower id comes first.

In [2]:
# The same three lists through the code: the order must be 3, 1, 2, 4, 5 and must match the verified reference.
toy = {"A": [1, 2, 3], "B": [2, 3, 4], "C": [3, 1, 5]}
print("fused order:", fu.fuse_rankings(toy, n=5), "| matches rank_fusion.fuse:", fu.check_matches_reference(toy, n=5))
pd.Series(fu.rrf_scores(toy)).sort_values(ascending=False).round(4).rename("fused score").to_frame()

fused order: [3, 1, 2, 4, 5] | matches rank_fusion.fuse: True


,fused score
3,0.0484
1,0.0325
2,0.0325
4,0.0159
5,0.0159


## 2 · How much the models agree

Rank fusion can only reward agreement that exists. For 3,000 sampled catalogue works, the table
counts how often two models' top-5 lists share at least one work and how large the overlap is.

In [3]:
# Pairwise overlap of the three top-5 lists on 3,000 sampled seed works (saved on first run).
agreement_path = PROCESSED / "book3_agreement.pkl"
if agreement_path.exists():
    agreement = pd.read_pickle(agreement_path)
else:
    agreement = fu.agreement_table(tables, work_ids, per_model=TOP_N, sample=3000, seed=SEED)
    agreement.to_pickle(agreement_path)
agreement.drop(columns=["model a", "model b"])

,mean overlap of top-5 sets,share of seeds with a common work
models,,
Directional (likes) vs Jaccard (likes),0.140,0.500
Directional (likes) vs Finished together (shelves),0.130,0.478
Jaccard (likes) vs Finished together (shelves),0.499,0.929
all three,NaN,0.464


## 3 · Tuning on validation

The agreed interface fixes what fusion receives: five works from each of the three models. Two
things were still open and were decided on validation readers, with the protocol from Book 2
(hide 20% of each reader's liked works, seed with the rest, score the fused list against the hidden
works at 5 and 10; agreement with Goodreads' own lists; coverage and popularity):

1. **Weights.** Equal weights, the directional score at half weight, or the directional score left
   out, since Book 2 showed it behaves like the popularity baseline.
2. **Depth.** Five from each model as agreed, against 10, 20 and 60 from each, to see what the
   agreement rule does with more candidates.

In [4]:
# Validation readers and the single models, as in Book 2.
val_readers = assignment.index[assignment == "validation"]
holdout_val = ev.holdout_split_for_readers(interactions, val_readers, work_ids, seed=SEED)
singles = fu.single_models(tables)
variants = fu.fusion_variants(tables)
print(f"validation readers scored: {len(holdout_val):,}")

validation readers scored: 4,082


In [5]:
# Weights, scored on validation readers (saved on first run).
grid_path = PROCESSED / "book3_weight_grid.pkl"
if grid_path.exists():
    weights_val = pd.read_pickle(grid_path)
else:
    weights_val = fu.weight_grid(tables, holdout_val, similar_books, work_ids, popularity, per_model=TOP_N)
    weights_val.to_pickle(grid_path)
weights_val[["precision@5", "precision@10", "hit@10", "ndcg@10", "Goodreads hit@10", "coverage", "median popularity (recs)"]]

,precision@5,precision@10,hit@10,ndcg@10,Goodreads hit@10,coverage,median popularity (recs)
model,,,,,,,
equal weights,0.1927,0.1859,0.7313,0.2344,0.2707,4744,9271.0
directional at 0.5,0.1928,0.1855,0.7295,0.2342,0.2900,4879,9271.0
directional at 0,0.2797,0.2233,0.7629,0.3062,0.2887,9909,1135.0


In [6]:
# Depth and the two-model variant, next to the single models, on validation readers (saved on first run).
val_path = PROCESSED / "book3_validation_results.pkl"
if val_path.exists():
    results_val = pd.read_pickle(val_path)
else:
    results_val = ev.evaluate_models({**singles, **variants}, holdout_val, similar_books, work_ids, popularity)
    results_val.to_pickle(val_path)
SHOW = ["precision@5", "precision@10", "hit@10", "ndcg@10", "Goodreads hit@10", "coverage"]
results_val[SHOW]

,precision@5,precision@10,hit@10,ndcg@10,Goodreads hit@10,coverage
model,,,,,,
Directional (likes),0.1368,0.0999,0.5017,0.1457,0.0510,439
Jaccard (likes),0.2034,0.1616,0.6779,0.2241,0.3150,6032
Finished together (shelves),0.1951,0.1565,0.6774,0.2175,0.3630,6562
Fused (top 5 each),0.1927,0.1859,0.7313,0.2344,0.2707,4744
Fused (top 10 each),0.1581,0.1436,0.6235,0.1883,0.3087,2539
Fused (top 20 each),0.1435,0.1136,0.5392,0.1613,0.3300,1316
Fused (top 60 each),0.1643,0.1339,0.6124,0.1848,0.3003,2123
"Fused, two Jaccard models only (top 5 each)",0.2797,0.2233,0.7629,0.3062,0.2887,9909


### Parameters kept

- **Five from each model, equal weights.** It is the interface the three models were built to. With
  all three models it is ahead of every single model on validation on precision at 10, recall, hit
  rate and NDCG, while slightly below the best single model on precision at 5.
- **Deeper inputs are not used.** Ten, twenty or sixty candidates per model lower every held-out
  measure. With more candidates the directional model's bestsellers appear in more lists and the
  agreement rule promotes them; the fused list drifts toward the popularity baseline.
- **The directional score is left out: the recommender fuses the two Jaccard models.** On validation,
  dropping it raises precision at 5 from 0.193 to 0.280, hit rate from 0.731 to 0.763 and NDCG from
  0.234 to 0.306, and lowers the median popularity of the recommendations from 9,271 to 1,135
  training readers. Half weight changes nothing. The three-model fusion stays in the test table as
  the comparison.

### Test readers, scored once

In [7]:
# Test readers: the fused recommenders, the three single models and the baselines on one table (saved on first run).
test_path = PROCESSED / "book3_test_results.pkl"
test_readers = assignment.index[assignment == "test"]
holdout_test = ev.holdout_split_for_readers(interactions, test_readers, work_ids, seed=SEED)
if test_path.exists():
    results_test = pd.read_pickle(test_path)
else:
    baselines = ev.baselines(train, work_ids, books, seed=SEED)
    compared = {**singles, "Fused (top 5 each)": variants["Fused (top 5 each)"],
                "Fused, two Jaccard models only (top 5 each)": variants["Fused, two Jaccard models only (top 5 each)"],
                **baselines}
    results_test = ev.evaluate_models(compared, holdout_test, similar_books, work_ids, popularity)
    results_test.to_pickle(test_path)
results_test[SHOW + ["median popularity (recs)"]]

,precision@5,precision@10,hit@10,ndcg@10,Goodreads hit@10,coverage,median popularity (recs)
model,,,,,,,
Directional (likes),0.1369,0.1008,0.5192,0.1459,0.0510,423,10073.0
Jaccard (likes),0.2080,0.1622,0.6881,0.2279,0.3150,6048,4196.0
Finished together (shelves),0.1991,0.1578,0.6845,0.2215,0.3630,6464,4304.0
Fused (top 5 each),0.1958,0.1890,0.7329,0.2371,0.2707,4737,9271.0
"Fused, two Jaccard models only (top 5 each)",0.2843,0.2302,0.7772,0.3128,0.2887,9813,1168.0
most popular,0.0796,0.0737,0.4223,0.0962,0.0000,36,10393.0
random,0.0005,0.0004,0.0039,0.0005,0.0013,17588,64.0
same-genre popular,0.0859,0.0648,0.4137,0.0969,0.0000,210,7682.0


## 4 · Visualization

One chart, one row per metric. The grey bar spans the three single models' test scores, with a dot
for each model; the filled diamond fuses all three models (top five from each) and the hollow
diamond is the recommender, the two Jaccard models fused. A diamond to the right of the bar is a
gain from fusion; a diamond inside the bar is an average of the models it combines.

In [8]:
# Where does fusion help and where does it average? One line per metric.
fc.fusion_range_chart(results_test)

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


Fusion lands beyond the bar on precision at 10, recall, hit rate and NDCG, where the three single
models are close together and the agreement rule adds what they share. It lands inside the bar on
precision at 5 and on agreement with Goodreads' lists, the two metrics where the directional score
sits far below the other two: there the rule averages a weak list in with two strong ones. The
hollow diamond shows what happens when that list is left out: a gain on every reader-level metric
and a much less mainstream catalogue, which is why it is the recommender.

## 5 · Results

- **The recommender, the two Jaccard models fused, is the best model on every reader-level measure.**
  On test readers it reaches precision at 5 of 0.284 (best single model 0.208), precision at 10 of
  0.230 (0.162), recall at 10 of 0.219 (0.160), hit rate of 0.777 (0.688) and NDCG of 0.313
  (0.228). It agrees with Goodreads' lists on 0.289 of seeds, below either Jaccard model alone
  (0.315 and 0.363).
- **Fusing the three agreed lists beats every single model on the reader-level measures except
  precision at 5.** On test readers the three-model fusion reaches precision at 10 of 0.189 (best
  single model 0.162), recall at 10 of 0.198 (0.160), hit rate of 0.733 (0.688) and NDCG of 0.237
  (0.228); its precision at 5 is 0.196 against 0.208 for Jaccard on likes.
- **It inherits the directional score's taste for bestsellers.** The median popularity of its
  recommendations is 9,271 training readers, close to the directional score's 10,073 and far from
  the 462 of the books readers actually liked; it recommends 4,737 distinct works against about
  6,000–6,500 for either Jaccard model, and agrees with Goodreads' lists less often (0.271 against
  0.315 and 0.363).
- **The two Jaccard models agree with each other; the directional score agrees with neither.** They
  share at least one top-5 work on 93% of seed works (mean overlap 0.50); the directional score
  shares one with either on about half of seeds (overlap 0.13–0.14); all three agree on 46%.
- **Leaving the directional score out improves every reader-level measure.** The two-model fusion
  recommends 9,813 distinct works at a median popularity of 1,168, against 4,737 at 9,271 for the
  three-model fusion. Half-weighting the directional score changes nothing; only removing it does.
- **More candidates per model make fusion worse.** Ten, twenty or sixty candidates per model lower
  precision at 5 on validation to 0.158, 0.144 and 0.164, because the directional model's
  bestsellers then appear in more lists and the agreement rule promotes them.

### Limitations

The comparison is made on the agreed interface: each model contributes its top five, so a fused
list is built from at most fifteen candidates per seed, while a single model's list for a reader is
built from its top sixty. The depth variants above show what equalising the inputs does; on this
data it lowers precision, because the extra candidates are mostly the directional score's
bestsellers. The held-out protocol and the Goodreads lists carry the same limits as in Book 2: hidden
likes are a proxy for what a reader would enjoy, and Goodreads' lists exclude same-author titles by
construction. The recommender was chosen on validation readers; test readers were scored once.

## 6 · The recommender

The dashboard calls one function: the two Jaccard models fused, with the same-author and same-series
filter applied, returning five `work_id`s for the selected work.

In [9]:
# The project's recommender: the two Jaccard models fused, filter on, five works.
related = co.related_works(books, work_ids)      # same-author and same-series works, removed at query time
jaccard_tables = {name: tables[name] for name in ["Jaccard (likes)", "Finished together (shelves)"]}

def recommend(work_id, n=TOP_N):
    """Top n recommendations for a work: the two Jaccard models' top five, fused by rank, same-author
    and same-series works removed."""
    return fu.recommend_fused(jaccard_tables, int(work_id), n=n, per_model=TOP_N, k=RRF_K,
                              exclude=frozenset(related.get(int(work_id), set())))


def show(title_start):
    seed = int(books[books["title"].str.startswith(title_start, na=False)].sort_values("ratings_count").iloc[-1]["work_id"])
    return pd.DataFrame({"recommendation": titles[recommend(seed)].to_numpy()}, index=pd.Index(range(1, TOP_N + 1), name=f"{titles[seed][:40]}"))

pd.concat([show("Gone Girl"), show("The Hobbit"), show("The Night Circus")], axis=1, keys=["Gone Girl", "The Hobbit", "The Night Circus"])

,Gone Girl,The Hobbit,The Night Circus
,recommendation,recommendation,recommendation
1,The Girl on the Train,Harry Potter and the Sorcerer's Stone (Harry P...,The Ocean at the End of the Lane
2,The Help,Harry Potter and the Prisoner of Azkaban (Harr...,Miss Peregrine’s Home for Peculiar Children (M...
3,"The Girl with the Dragon Tattoo (Millennium, #1)","A Game of Thrones (A Song of Ice and Fire, #1)",Water for Elephants
4,The Fault in Our Stars,Harry Potter and the Goblet of Fire (Harry Pot...,The Book Thief
5,The Lovely Bones,Harry Potter and the Deathly Hallows (Harry Po...,Gone Girl


## 7 · Files written

| File | Contents |
|---|---|
| `data/processed/book3_agreement.pkl` | pairwise overlap of the three models' top-5 lists on 3,000 seed works |
| `data/processed/book3_weight_grid.pkl` | the three weight settings scored on validation readers |
| `data/processed/book3_validation_results.pkl` | single models and fusion variants on validation readers |
| `data/processed/book3_test_results.pkl` | single models, the two fused recommenders and the baselines on test readers |

The recommender itself needs only the two Jaccard neighbour tables from Book 2 and `scripts/fusion.py`.